# Official CODI teacher-trace selector specificity

## Goal

Determine whether R-KV selects teacher trace positions with more transferable low-rank key/value signal than uniform or seeded-random selection. Every arm uses the same accuracy-gated official CODI checkpoint, 5,000 examples, split assignments, model forward passes, student latent states, and shuffled-pairing nulls.

## 1. Choose the run scope

Run the audit first, then the complete matched collection and CPU analysis. The collection is restartable from an atomic Drive checkpoint. Keep the Colab runtime connected while a cell is active because Drive persistence does not make Colab itself a background service.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the pushed immutable commit before the final run.
REPO_DIR = "/content/latent-reasoning"
DRIVE_ROOT = "/content/drive/MyDrive/CODI_KAVA"

RUN_AUDIT = True
RUN_COLLECTION = True
RUN_ANALYSIS = True

EXAMPLES = 5000
DATA_SEED = 1
RANDOM_SELECTOR_SEEDS = [101, 211, 307, 401]
BATCH_SIZE = 16
SHUFFLE_REPEATS = 4
SAVE_EVERY = 1000
PRECISION = "bfloat16"

GATE_RANK = 4
SELECTOR_SIGNAL_MARGIN = 0.01
SELECTOR_WIN_FRACTION = 0.60

## 2. Mount Drive and install the pinned environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import datetime
import json
import os
import pathlib
import subprocess
import sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("Pin RUN_COMMIT before the final collection to:", commit)

## 3. Verify GPU, code contracts, and the official accuracy gate

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a Colab GPU runtime"
gpu_name = torch.cuda.get_device_name(0)
print("Torch:", torch.__version__)
print("GPU:", gpu_name)
if "A100" not in gpu_name:
    print("Warning: this workflow runs on the current GPU, but the time estimate assumed an A100.")

subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_official_codi.py",
        "tests/test_official_codi_kv.py",
        "tests/test_kv_compress.py",
        "tests/test_kv_cross_subspace.py",
        "tests/test_kv_reduced_rank.py",
        "tests/test_kv_selector_specificity.py",
    ],
    cwd=REPO_DIR,
    check=True,
)

official_root = pathlib.Path(DRIVE_ROOT) / "outputs" / "official_codi_gpt2"
gate_candidates = sorted(official_root.glob("eval/revision_*/full_gsm8k/summary.json"))
passed = []
for candidate in gate_candidates:
    payload = json.loads(candidate.read_text())
    gate = payload.get("accuracy_gate", payload.get("gate"))
    status = gate if isinstance(gate, str) else (gate or {}).get("status")
    if status == "passed":
        passed.append(candidate)
assert passed, "Run colab_official_codi_validation.ipynb until full GSM8K passes"
REPRODUCTION_SUMMARY = passed[-1]
print("Passed reproduction summary:", REPRODUCTION_SUMMARY)

## 4. Define Drive-persistent execution

In [ ]:
OUTPUT_DIR = pathlib.Path(DRIVE_ROOT) / "outputs" / "official_codi_selector_specificity" / "n5000_seed1"
REPORT_ROOT = pathlib.Path(DRIVE_ROOT) / "reports" / "official_codi_selector_specificity"
LOG_ROOT = pathlib.Path(DRIVE_ROOT) / "logs" / "official_codi_selector_specificity"
for path in (OUTPUT_DIR, REPORT_ROOT, LOG_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def run_persisted(cmd, log_name):
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, cmd)), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, cmd))} ===\n")
        process = subprocess.Popen(
            list(map(str, cmd)), cwd=REPO_DIR, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
        log.flush()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)
    return code

def collection_command(output_dir, examples, audit_only=False):
    cmd = [
        sys.executable, "-u", "scripts/collect_official_codi_selector_subspaces.py",
        "--config", "configs/official_codi_gpt2.yaml",
        "--reproduction-summary", str(REPRODUCTION_SUMMARY),
        "--output-dir", str(output_dir),
        "--examples", str(examples),
        "--batch-size", str(BATCH_SIZE),
        "--shuffle-repeats", str(SHUFFLE_REPEATS),
        "--save-every", str(SAVE_EVERY),
        "--precision", PRECISION,
        "--device", "cuda",
        "--seed", str(DATA_SEED),
        "--random-selector-seeds", ",".join(map(str, RANDOM_SELECTOR_SEEDS)),
    ]
    if audit_only:
        cmd.append("--audit-only")
    return cmd

## 5. Run the one-batch matched-selector audit

This checks tensor shapes, finite values, selector coverage, representative indices, and pairwise index agreement without allocating the multi-selector moment collection.

In [ ]:
from IPython.display import JSON, Markdown, display

AUDIT_DIR = pathlib.Path(DRIVE_ROOT) / "outputs" / "official_codi_selector_specificity" / "audit_seed1"
if RUN_AUDIT:
    run_persisted(collection_command(AUDIT_DIR, BATCH_SIZE, audit_only=True), "selector_audit.log")
    audit = json.loads((AUDIT_DIR / "selection_audit.json").read_text())
    assert audit["student_latent_shape"][1:] == [12, 12, 6, 64]
    assert audit["finite_student_keys"]
    assert set(audit["selectors"]) == {"rkv", "uniform", *[f"random_seed{s}" for s in RANDOM_SELECTOR_SEEDS]}
    for selector, values in audit["selectors"].items():
        assert values["selected_valid_fraction"] > 0.0, f"{selector} selected no valid targets"
        assert values["finite_teacher_keys"] and values["finite_teacher_values"]
    display(JSON(audit))
else:
    print("Audit skipped")

## 6. Collect the complete matched 5,000-example statistics

The collector performs one teacher/student forward pass per batch and accumulates all six selector arms together. Rerun this cell after a disconnect to continue from the latest atomic Drive checkpoint.

In [ ]:
if RUN_COLLECTION:
    run_persisted(collection_command(OUTPUT_DIR, EXAMPLES), "collect_n5000_seed1.log")
    manifest = json.loads((OUTPUT_DIR / "collection_manifest.json").read_text())
    assert manifest["state"] == "complete"
    assert manifest["processed_examples"] == EXAMPLES
    assert manifest["random_selector_seeds"] == RANDOM_SELECTOR_SEEDS
    print(json.dumps({k: manifest[k] for k in ("state", "processed_examples", "selectors", "indices_sha256")}, indent=2))
else:
    print("Collection skipped")

## 7. Run the preregistered selector-specificity analysis

The primary score is held-out rank-4 R² for the actual pairing minus held-out rank-4 R² for the same selector under teacher-example shuffling. R-KV must beat both uniform selection and the per-group median of four random selectors by at least 0.01 R² in at least 60 percent of matched layer-head-position groups.

In [ ]:
REPORT_PATH = REPORT_ROOT / "official_codi_n5000_seed1_selector_specificity.json"
if RUN_ANALYSIS:
    run_persisted(
        [
            sys.executable, "scripts/analyze_kv_selector_specificity.py",
            "--statistics", str(OUTPUT_DIR),
            "--output", str(REPORT_PATH),
            "--gate-rank", str(GATE_RANK),
            "--selector-signal-margin", str(SELECTOR_SIGNAL_MARGIN),
            "--selector-win-fraction", str(SELECTOR_WIN_FRACTION),
        ],
        "analyze_n5000_seed1.log",
    )
    display(Markdown(REPORT_PATH.with_suffix(".md").read_text()))
else:
    print("Analysis skipped")

## 8. Checks and next decision

In [ ]:
print("\nDurable collection")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name, f"{path.stat().st_size / (1024 ** 2):.1f} MiB")

print("\nDurable reports")
for path in sorted(REPORT_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(REPORT_ROOT), f"{path.stat().st_size / 1024:.1f} KiB")

if REPORT_PATH.is_file():
    result = json.loads(REPORT_PATH.read_text())
    print("\nGate:", result["gate"]["status"])

display(Markdown("""
### Interpretation boundary

A positive gate supports the claim that R-KV selects teacher trace positions with more transferable linear KV signal than matched uniform and random selectors. It does not show that the subspace causes correct answers or improves training accuracy. If the gate passes, the next experiment is a compute-matched downstream comparison of full targets, learned rank-4 targets, and random rank-4 targets. If it fails, R-KV token selection should not be treated as the source of the previously observed spectral signal.
"""))